# WorkspaceBench

A look at the banks behind `src/domains/lm/data/workspacebench.py`.

[WorkspaceBench](https://github.com/camilablank/workspace-bench) (Blank, Bhatia, Ong and Nanda,
2026) asks whether an activation-reading lens surfaces what a model computed but never wrote. Each
item is a prompt whose hidden intermediates are known *by construction* — the model must have held
`carnaval` to continue that sentence, and never says it — so a readout can be graded against
something other than a human's impression of it.

Nothing here is downloaded: each bank is fetched from GitHub at construction, pinned to a commit,
and held in memory.

In [1]:
import sys
from collections import Counter

sys.path.insert(0, "..")  # this notebook sits in notebooks/, the package is at the repo root

import json

import pandas as pd

from src.domains.lm.data.workspacebench import COMMIT, FAMILIES, WorkspaceBench, fetch_bank, item_loader

print(f"{len(FAMILIES)} banks, pinned to {COMMIT[:7]}")
FAMILIES

26 banks, pinned to 92d763e


('agentic_misalignment',
 'arithmetic_intermediates',
 'association',
 'basic_readout',
 'basic_readout_mt',
 'brew_intermediates',
 'buggy_code',
 'chain_intermediates',
 'conjunctive_association',
 'directed_modulation',
 'hallucination',
 'jailbreak_recognition',
 'moral_rationale',
 'multi_concept_directed_modulation',
 'multihop',
 'multihop_mt',
 'multilingual',
 'multilingual_mt',
 'multilingual_multihop',
 'multilingual_typo',
 'poetry',
 'relational_multihop',
 'role_bound_association',
 'typo',
 'typo_mt',
 'user_modeling')

## One bank

`WorkspaceBench(family)` is a map-style `torch.utils.data.Dataset`. The bank's **gate** — the
sentence a judge scores a readout against — sits on the dataset rather than on every row, because
it is one sentence for the whole bank.

In [2]:
bank = WorkspaceBench("association")
print(bank)
print()
print(bank.gate)

WorkspaceBench('association', n=100, ref=92d763e)

Qwen3.6-27B behavioral: >=8/10 correct at temp 0.7 (greedy-verified first); association scored against association_aliases.json (phrase-aware, echo-guarded); poetry scored by committed rhyme


## One item

An item comes back as the dict it is in the bank, unconverted. `intermediates` is the thing the
model has to have computed; `readout` says where a lens should be asked for it.

In [3]:
print(json.dumps(bank[0], indent=2, ensure_ascii=False))

{
  "name": "pt-carnaval",
  "prompt": "Os tambores começaram na avenida ao anoitecer, e as fantasias cobertas de plumas dançaram até o amanhecer.",
  "intermediates": [
    "carnaval"
  ],
  "source": "staging_b3",
  "consistency": 1.0,
  "gate_variant": "referent",
  "eval_render": "plain",
  "readout": {
    "kind": "final_prompt_token",
    "offsets": [
      -1
    ]
  }
}


## The intermediates are the point

The prompt never contains the word in `intermediates`. A readout that names it is reading the
model, not the prompt — which is exactly the claim the prompt-only baseline exists to attack.

In [4]:
rows = [{"item": item["name"],
         "prompt": item["prompt"][:64] + "…",
         "intermediates": ", ".join(item["intermediates"]),
         "in the prompt?": any(i.lower() in item["prompt"].lower() for i in item["intermediates"])}
        for item in list(bank)[:8]]
pd.DataFrame(rows)

,item,prompt,intermediates,in the prompt?
0,pt-carnaval,"Os tambores começaram na avenida ao anoitecer,...",carnaval,False
1,chess,Neither of them had spoken in forty minutes wh...,chess,False
2,child-h,The crayon drawings on the fridge had started ...,child,False
3,death-h,The flowers on the table had been changed twic...,death,False
4,film-h,"The lights went down and the logo came up, and...",film,False
5,lie,"He couldn't meet her eyes when she asked, and ...",lie,False
6,nasa,The countdown hit zero and the rumble under he...,NASA,False
7,neighbors,She knew what time his car left and what night...,neighbors,False


## 26 banks, 26 schemas

This is why the loader converts nothing. `association` carries `prompt`/`intermediates`/`readout`,
`moral_rationale` carries `stimulus`/`question`/`gold_reasons`, `agentic_misalignment` carries
`system`/`text`. A common row shape would be one the benchmark does not have.

In [5]:
for family in ("association", "moral_rationale", "agentic_misalignment", "buggy_code"):
    keys = sorted(fetch_bank(family)["items"][0])
    print(f"{family:24s} {', '.join(keys[:9])}{' …' if len(keys) > 9 else ''}")

association              consistency, eval_render, gate_variant, intermediates, name, prompt, readout, source
moral_rationale          answer_format, answer_split, commit_answer, commit_direction, commit_precise, domain, gate_aligned, gate_committed_token, gate_pass …


agentic_misalignment     arm, category, descriptor, group, id, label, misalignment_rate, prefill, rollout_pick …
buggy_code               bug_line, category, cause, code, comment_line, consequence_class, depth, difficulty, failure_mode …


## Every bank, by size

One fetch per bank — 26 small JSON files. The banks with a `gate` are the ones whose items a judge
scores directly; four are bare lists on disk and come back normalised to `{"items": [...]}`.

In [6]:
banks = {family: fetch_bank(family) for family in FAMILIES}

sizes = pd.DataFrame(
    [{"bank": family,
      "items": len(bank["items"]),
      "gate": "yes" if "gate" in bank else "—",
      "fields on item 0": len(bank["items"][0])}
     for family, bank in banks.items()]
).sort_values("items", ascending=False).reset_index(drop=True)

print(f"{len(sizes)} banks, {sizes['items'].sum()} items")
sizes

26 banks, 3057 items


,bank,items,gate,fields on item 0
0,arithmetic_intermediates,596,—,27
1,moral_rationale,200,—,25
2,hallucination,149,—,7
3,chain_intermediates,120,yes,14
4,role_bound_association,100,—,18
5,relational_multihop,100,—,9
6,poetry,100,yes,8
7,multilingual_typo,100,yes,17
8,multilingual_mt,100,yes,17
9,multilingual,100,yes,10


3,057 items over 26 banks. The write-up's headline is 3,356 over 27: the difference is
`jlens_concept_pr` (per-layer readouts, not items) and `baselines` (scores), which live under
`evals/` too and are not item banks.

## What the banks ask

A prompt from each of a few banks, to show how far apart they are.

In [7]:
for family in ("multihop", "typo", "poetry", "arithmetic_intermediates"):
    item = banks[family]["items"][0]
    print(f"── {family} ──")
    print(item["prompt"].strip()[:300])
    print(f"   intermediates: {item.get('intermediates')}")
    print()

── multihop ──
Fact: The chemical symbol for the element with atomic number 26 is
   intermediates: ['iron']

── typo ──
Her birthday falls on the last day of Febuary
   intermediates: ['February']

── poetry ──
A rhyming couplet:
The captain pointed at the route ahead,
And told his crew to follow where he'd
   intermediates: ['led']

── arithmetic_intermediates ──
Compute abs(696 - 997) * 16. Reply with only the final number, nothing else.
   intermediates: ['301']



## Batching

`item_loader` uses `collate_items`, so a batch stays a list of dicts. The default collate would try
to stack a field that is a string in one bank, a list of strings in another, and absent in a third.

In [8]:
batch = next(iter(item_loader(bank, batch_size=4)))
print(type(batch), len(batch))
[item["name"] for item in batch]

<class 'list'> 4


['pt-carnaval', 'chess', 'child-h', 'death-h']

## What is not here

The benchmark scores a **readout** — an activation-to-text tool's claim about what was in the
workspace — and this lab produces none yet. So what is usable today is the prompts, the
intermediates and the gates; the scoring half (`wsbench plan | produce | judge | report`) lives
upstream.

Worth remembering from the write-up before trusting any number this produces later: a prompt-only
baseline, reasoning about the workspace from context with no access to activations at all, does
well on many of these banks.